# Humanitarian AI (HAI) Model Training
## Fine-tuning Llama 3.2 3B for Humanitarian Expertise

**Dataset**: 93 humanitarian examples (79 train, 9 val, 5 test)  
**Method**: LoRA fine-tuning with 4-bit quantization  
**Expected time**: 15-30 minutes on T4 GPU  
**Cost**: Free (or $12/month Colab Pro)

---

### Setup Instructions:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Upload files**: Use the file upload in Cell 2
3. **Run all cells**: Runtime → Run all
4. **Download model**: After training completes (Cell 7)


In [ ]:
# Cell 1: Verify GPU and setup environment
!nvidia-smi
import os
os.makedirs('/content/hai-cd', exist_ok=True)
os.chdir('/content/hai-cd')
print("\n✅ Working directory:", os.getcwd())
print("✅ GPU detected - ready for training!")

In [ ]:
# Cell 2: Upload required files
from google.colab import files
import json

print("📤 Please upload the following files from hai-cd directory:")
print("   1. train_dataset.json")
print("   2. val_dataset.json")
print("   3. test_dataset.json")
print("   4. config.yaml")
print("   5. train.py")
print("   6. app.py (optional - for demo)")
print("\n👇 Click 'Choose Files' below and select all files:")

uploaded = files.upload()

# Verify uploads
required_files = ['train_dataset.json', 'val_dataset.json', 'test_dataset.json', 'config.yaml', 'train.py']
missing = [f for f in required_files if f not in uploaded]

if missing:
    print(f"\n❌ Missing files: {missing}")
    print("Please upload all required files and re-run this cell.")
else:
    print("\n✅ All required files uploaded!")
    with open('train_dataset.json', 'r') as f:
        train_data = json.load(f)
    print(f"✅ Training samples: {len(train_data)}")

In [ ]:
# Cell 3: Install dependencies
print("📦 Installing ML dependencies (this may take 3-5 minutes)...\n")

!pip install -q transformers>=4.45.0 torch>=2.0.0 peft>=0.12.0 
!pip install -q accelerate>=0.34.0 bitsandbytes>=0.42.0
!pip install -q datasets>=2.20.0 pyyaml>=6.0.0 tqdm>=4.66.0
!pip install -q gradio>=4.44.0

print("\n✅ Dependencies installed successfully!")

# Verify key imports
import transformers
import torch
import peft
print(f"\n📊 Package versions:")
print(f"   Transformers: {transformers.__version__}")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 4: Verify training configuration
import yaml

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("⚙️  Training Configuration:\n")
print(f"Base Model: {config['model']['base_model']}")
print(f"Quantization: {config['model']['quantization']}")
print(f"LoRA rank: {config['lora']['r']}")
print(f"\nTraining:")
print(f"  Epochs: {config['training']['num_epochs']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"\nBudget: ${config['budget']['max_cost_usd']}")

# Show dataset info
with open('dataset_summary.json', 'w') as f:
    summary = {
        "total_samples": len(train_data),
        "platform": "Google Colab",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    }
    json.dump(summary, f, indent=2)

print("\n✅ Configuration verified - ready to train!")

In [ ]:
# Cell 5: START TRAINING (Main Training Cell)
import time

print("🚀 Starting model training...\n")
print("📊 This will take approximately 15-30 minutes")
print("💡 You can monitor progress in the output below")
print("\n" + "="*60 + "\n")

start_time = time.time()

# Run training script
!python train.py

elapsed = time.time() - start_time
print("\n" + "="*60)
print(f"\n✅ Training completed in {elapsed/60:.1f} minutes!")
print("\n📁 Model saved to: ./humanitarian-model/")
print("\n📊 Check training_metrics.json for detailed results")

In [ ]:
# Cell 6: View training results
import json
import matplotlib.pyplot as plt

# Load metrics
with open('training_metrics.json', 'r') as f:
    metrics = json.load(f)

print("📊 Training Results:\n")
print(f"Final training loss: {metrics.get('final_train_loss', 'N/A')}")
print(f"Final validation loss: {metrics.get('final_val_loss', 'N/A')}")
print(f"Training time: {metrics.get('total_time', 'N/A')}")

# Plot loss curves if available
if 'loss_history' in metrics:
    plt.figure(figsize=(10, 5))
    plt.plot(metrics['loss_history'], label='Training Loss')
    if 'val_loss_history' in metrics:
        plt.plot(metrics['val_loss_history'], label='Validation Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('HAI Model Training Progress')
    plt.legend()
    plt.grid(True)
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ Training curves saved to: training_curves.png")

# Show model size
!du -sh humanitarian-model/
print("\n✅ Model ready for deployment!")

In [ ]:
# Cell 7: Download trained model
from google.colab import files
import shutil

print("📦 Preparing model for download...\n")

# Create archive
!tar -czf humanitarian-model.tar.gz humanitarian-model/
!tar -czf training-outputs.tar.gz training_metrics.json training_curves.png dataset_summary.json

print("✅ Archives created")
print("\n📥 Downloading files (this may take a few minutes)...\n")

# Download model
files.download('humanitarian-model.tar.gz')
files.download('training-outputs.tar.gz')

print("\n✅ Downloads complete!")
print("\n📁 You now have:")
print("   1. humanitarian-model.tar.gz - The trained model")
print("   2. training-outputs.tar.gz - Metrics and visualizations")
print("\n🎉 Training complete! Next: Test with Gradio demo app")

In [ ]:
# Cell 8 (OPTIONAL): Launch Gradio demo
print("🎨 Launching Gradio demo app...\n")
print("⚠️  Note: This will load the model into memory (may take 2-3 minutes)\n")

# Run demo app
!python app.py

print("\n✅ Demo app is running!")
print("\n💡 Click the public URL above to test your humanitarian AI model")
print("\n📝 Try questions like:")
print("   - What are the statistics on humanitarian funding?")
print("   - How should responders address natural disasters?")
print("   - What is GDPR compliance in humanitarian context?")

---

## Next Steps

1. **Extract model locally**:
   ```bash
   tar -xzf humanitarian-model.tar.gz
   ```

2. **Run Petri auditing** (from `hai/` directory):
   ```bash
   cd ../hai
   python run_audit.py --model-path ../hai-cd/humanitarian-model
   ```

3. **Deploy** (see HAI_FINAL_SUMMARY.md for deployment options)

---

## Troubleshooting

**Out of memory**: Reduce `batch_size` in config.yaml to 2  
**Slow training**: Verify GPU is enabled (check Cell 1)  
**Session timeout**: Upgrade to Colab Pro for longer sessions  

**Total cost**: $0 (free tier) or $12/month (Colab Pro)  
**Training time**: 15-30 minutes on T4, 10-15 minutes on A100
